In [0]:
# 1. Configuration
catalog = "capstone6_telecom"
bronze_schema = "bronze"
silver_schema = "silver"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{silver_schema}")

In [0]:
# 2. Imports, helpers and the data-quality engine
from functools import reduce
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Rejected / warning tables hold the current state only (rebuilt on every run, no run history)
rejected_table = f"{catalog}.{silver_schema}.silver_rejected_records"
warning_table = f"{catalog}.{silver_schema}.silver_warning_records"

AUDIT_COLS = ["_rescued_data", "_source_system", "_source_feed", "_source_table", "_source_file", "_ingested_at"]
REJECT, WARN = "REJECT", "WARN"   # REJECT = kept out of Silver, WARN = kept in Silver and flagged

def read_bronze(table_name):
    return spark.table(f"{catalog}.{bronze_schema}.{table_name}")

def silver_table(name):
    return spark.table(f"{catalog}.{silver_schema}.{DATASETS[name]['silver']}")

def save_silver(df, table_name):
    (df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{catalog}.{silver_schema}.{table_name}"))
    print(f"Saved {catalog}.{silver_schema}.{table_name}")

def raw(c):
    return F.col("_raw").getField(c)            # original Bronze value, as text

def blank(c):
    return F.nullif(F.trim(raw(c)), F.lit("")).isNull()

def issue(level, category, check, column, cond, value=None):
    """One issue struct when cond is true, otherwise NULL."""
    return F.when(cond, F.struct(
        F.lit(level).alias("level"),
        F.lit(category).alias("category"),
        F.lit(check).alias("check"),
        F.lit(column).alias("column"),
        (raw(column) if value is None else value).cast("string").alias("value")))

def is_rejected():
    return F.exists("_issues", lambda x: x["level"] == REJECT)

def prepare(cfg):
    """Standardise text, keep raw values, convert types and rank duplicates. Bronze is only read."""
    df = read_bronze(cfg["bronze"])
    cols = df.columns
    src = [c for c in cols if c not in AUDIT_COLS]
    audit = [c for c in cols if c in AUDIT_COLS]
    key, cats = cfg["key"], cfg.get("categories", {})

    std = {c: F.nullif(F.trim(F.col(c).cast("string")), F.lit("")) for c in src}
    std.update({c: F.upper(std[c]) for c in cats})
    lineage = [F.col(c) for c in ("_source_file", "_source_table") if c in cols]

    df = df.select(
        *[std[c].alias(c) for c in src], *audit,
        F.struct(*[F.col(c).cast("string").alias(c) for c in src]).alias("_raw"),
        F.to_json(F.struct(*[F.col(c) for c in cols])).alias("_record"),
        F.sha2(F.concat_ws("||", *[F.coalesce(F.col(c).cast("string"), F.lit("<NULL>")) for c in src]), 256).alias("_row_hash"),
        F.coalesce(*lineage, F.lit(cfg["bronze"])).alias("_lineage"))

    # Type conversion: try_cast returns NULL on failure; failures are flagged in row_issues
    if cfg.get("casts"):
        df = df.withColumns({c: F.expr(f"try_cast(`{c}` AS {t})") for c, t in cfg["casts"].items()})

    # Deterministic dedup: exact copies keep the first delivery;
    # different versions of the same key keep the latest ingested (tie-break: lineage, row hash)
    w_exact = Window.partitionBy("_row_hash").orderBy(F.col("_ingested_at").asc_nulls_last(), F.col("_lineage"))
    df = df.withColumn("_exact_first", F.row_number().over(w_exact) == 1)
    w_key = (Window.partitionBy(key, "_exact_first")
             .orderBy(F.col("_ingested_at").desc_nulls_last(), F.col("_lineage").desc(), F.col("_row_hash")))
    df = df.withColumn("_key_rn", F.row_number().over(w_key))
    return df, src, audit

def row_issues(cfg, src):
    key, mandatory = cfg["key"], set(cfg.get("mandatory", []))
    out = []
    # 1. Null checks
    for c in src:
        if c == key:
            out.append(issue(REJECT, "NULL_CHECK", "MISSING_BUSINESS_KEY", c, blank(c)))
        elif c in mandatory:
            out.append(issue(REJECT, "NULL_CHECK", "MISSING_MANDATORY_FIELD", c, blank(c)))
        else:
            out.append(issue(WARN, "NULL_CHECK", "MISSING_OPTIONAL_FIELD", c, blank(c)))
    # 2. Duplicate checks
    out.append(issue(REJECT, "DUPLICATE", "EXACT_DUPLICATE_ROW", key, ~F.col("_exact_first")))
    out.append(issue(REJECT, "DUPLICATE", "DUPLICATE_BUSINESS_KEY", key,
                     F.col(key).isNotNull() & F.col("_exact_first") & (F.col("_key_rn") > 1)))
    # 3. Data type checks (6. invalid timestamps for DATE/TIMESTAMP columns)
    for c, t in cfg.get("casts", {}).items():
        is_time = t.upper() in ("DATE", "TIMESTAMP")
        out.append(issue(REJECT, "TIMESTAMP" if is_time else "DATA_TYPE",
                         "INVALID_TIMESTAMP" if is_time else f"INVALID_{t.upper()}",
                         c, F.col(c).isNull() & ~blank(c)))
    # 4. Numeric validation
    for c in cfg.get("non_negative", []):
        out.append(issue(REJECT, "NUMERIC_RANGE", "NEGATIVE_VALUE", c, F.col(c) < 0))
    for c in cfg.get("percent", []):
        out.append(issue(REJECT, "NUMERIC_RANGE", "PERCENT_OUT_OF_RANGE", c, (F.col(c) < 0) | (F.col(c) > 100)))
    # 5. Categorical validation
    for c, allowed in cfg.get("categories", {}).items():
        out.append(issue(WARN, "CATEGORICAL", "UNEXPECTED_CATEGORY", c, F.col(c).isNotNull() & ~F.col(c).isin(allowed)))
    # 6. Timestamp order
    for later, earlier, check in cfg.get("time_order", []):
        out.append(issue(REJECT, "TIMESTAMP", check, later, F.col(later) < F.col(earlier)))
    return out

def add_references(df, refs):
    """7. Referential integrity: refs = [(column, ref_df, ref_key, check_name)] against clean Silver masters."""
    exprs = []
    for i, (c, ref_df, ref_key, check) in enumerate(refs):
        ref = ref_df.select(F.col(ref_key).alias(f"_rk{i}")).distinct().withColumn(f"_ref_ok{i}", F.lit(True))
        df = df.join(F.broadcast(ref), F.col(c) == F.col(f"_rk{i}"), "left")
        exprs.append(issue(REJECT, "REFERENTIAL", check, c, F.col(c).isNotNull() & F.col(f"_ref_ok{i}").isNull()))
    return df, exprs

def add_outliers(df, metrics, k):
    """8. Tukey IQR fences, computed on records that passed all REJECT checks."""
    q = (df.filter(~is_rejected())
           .agg(*[F.percentile_approx(m, [0.25, 0.75], 10000).alias(m) for m in metrics])
           .first())
    fences = {m: (q[m][0] - k * (q[m][1] - q[m][0]), q[m][1] + k * (q[m][1] - q[m][0])) for m in metrics}
    for m, (lo, hi) in fences.items():
        print(f"  outlier fence {m}: [{lo:.2f}, {hi:.2f}]  (Q1={q[m][0]}, Q3={q[m][1]})")
    new = F.filter(F.array(*[issue(WARN, "OUTLIER", "IQR_OUTLIER", m, (F.col(m) < lo) | (F.col(m) > hi))
                             for m, (lo, hi) in fences.items()]), lambda x: x.isNotNull())
    return df.withColumn("_issues", F.concat(F.col("_issues"), new))

def finalize(df, cfg, src, audit):
    """Write the clean Silver table and collect rejected / warning issue rows."""
    warn_checks = F.array_distinct(F.transform(F.filter("_issues", lambda x: x["level"] == WARN), lambda x: x["check"]))
    save_silver(df.filter(~is_rejected()).select(*src, *audit, warn_checks.alias("dq_warning_checks")), cfg["silver"])

    issues = (df.withColumn("_rec_rejected", is_rejected())
        .select(F.col(cfg["key"]).alias("business_key_value"), "_ingested_at", "_record", "_rec_rejected",
                F.explode("_issues").alias("_i"))
        # every detected issue; the write step stores REJECT reasons of rejected records
        # and WARN findings of kept records in separate tables
        .select(
            F.when(F.col("_rec_rejected"), F.lit("REJECTED")).otherwise(F.lit("VALID")).alias("record_status"),
            F.lit(cfg["bronze"]).alias("source_table"),
            F.lit(cfg["silver"]).alias("target_table"),
            F.lit(cfg["key"]).alias("business_key_column"),
            "business_key_value",
            F.col("_i.level").alias("issue_level"),
            F.col("_i.category").alias("issue_category"),
            F.col("_i.check").alias("check_name"),
            F.col("_i.column").alias("issue_column"),
            F.col("_i.value").alias("issue_value"),
            F.col("_ingested_at").alias("source_ingested_at"),
            F.col("_record").alias("source_record_json"),
            F.current_timestamp().alias("checked_at")))
    issue_frames.append(issues)

def run_dataset(name, extra=None):
    cfg = DATASETS[name]
    print(f"Processing {cfg['bronze']}")
    df, src, audit = prepare(cfg)
    # Referential checks come from the rule config and are resolved against the clean Silver masters
    refs = [(c, silver_table(ref_ds), ref_key, check) for c, ref_ds, ref_key, check in cfg.get("refs", [])]
    df, exprs = add_references(df, refs)
    if extra is not None:
        df, more = extra(df)
        exprs = exprs + more
    df = df.withColumn("_issues", F.filter(F.array(*(row_issues(cfg, src) + exprs)), lambda x: x.isNotNull()))
    if cfg.get("outlier_metrics"):
        df = add_outliers(df, cfg["outlier_metrics"], OUTLIER_IQR_K)
    finalize(df, cfg, src, audit)

In [0]:
# 3. Data-quality rules per dataset
# Rules come from the Bronze schemas and values observed during profiling. No SLA thresholds are used.
# Categorical domains = the clean values present in Bronze; anything else becomes a WARN, not a reject.
# Mandatory = business key, parent reference and the main event timestamp (records unusable without them).
DATASETS = {
    "region": {
        "bronze": "bronze_region_lookup", "silver": "silver_region_lookup", "key": "region_id",
        "mandatory": ["region_id"],
        "categories": {"active_flag": ["Y", "N"]},
    },
    "tower": {
        "bronze": "bronze_tower_master", "silver": "silver_tower_master", "key": "tower_id",
        "mandatory": ["tower_id", "region_id"],
        "refs": [("region_id", "region", "region_id", "INVALID_REGION_REFERENCE")],
        "casts": {"latitude": "DOUBLE", "longitude": "DOUBLE", "activation_date": "DATE"},
        "categories": {"site_type": ["BASIC", "STANDARD", "PREMIUM"],
                       "network_generation": ["4G", "5G"],
                       "tower_status": ["ACTIVE", "INACTIVE", "PENDING"]},
    },
    "topology": {
        "bronze": "bronze_network_topology_master", "silver": "silver_network_topology_master", "key": "cell_id",
        "mandatory": ["cell_id", "tower_id", "region_id"],
        "refs": [("tower_id", "tower", "tower_id", "INVALID_TOWER_REFERENCE"),
                 ("region_id", "region", "region_id", "INVALID_REGION_REFERENCE")],
        "casts": {"planned_capacity": "DOUBLE", "commissioned_date": "DATE"},
        "categories": {"technology": ["4G_LTE", "5G_NR"],   # repaired source (2026-11-04); was TECHN_1..TECHN_7
                       "active_flag": ["Y", "N"]},
    },
    "kpi": {
        "bronze": "bronze_network_kpi_measurements", "silver": "silver_network_kpi_measurements", "key": "measurement_id",
        "mandatory": ["measurement_id", "tower_id", "measurement_ts"],
        "refs": [("tower_id", "tower", "tower_id", "INVALID_TOWER_REFERENCE"),
                 ("cell_id", "topology", "cell_id", "INVALID_CELL_REFERENCE"),
                 ("region_id", "region", "region_id", "INVALID_REGION_REFERENCE")],
        "casts": {"measurement_ts": "TIMESTAMP", "latency_ms": "DOUBLE", "packet_loss_pct": "DOUBLE",
                  "throughput_mbps": "DOUBLE", "active_sessions": "BIGINT", "availability_pct": "DOUBLE",
                  "ingestion_ts": "TIMESTAMP"},
        "non_negative": ["latency_ms", "throughput_mbps", "active_sessions"],
        "percent": ["packet_loss_pct", "availability_pct"],
        "categories": {"network_generation": ["4G", "5G"]},
        "outlier_metrics": ["latency_ms", "packet_loss_pct", "throughput_mbps", "active_sessions", "availability_pct"],
    },
    "outage": {
        "bronze": "bronze_network_outage_events", "silver": "silver_network_outage_events", "key": "outage_id",
        "mandatory": ["outage_id", "tower_id", "outage_start_ts"],
        "refs": [("tower_id", "tower", "tower_id", "INVALID_TOWER_REFERENCE")],
        "casts": {"outage_start_ts": "TIMESTAMP", "outage_end_ts": "TIMESTAMP", "reported_ts": "TIMESTAMP",
                  "affected_cells": "INT"},
        "categories": {"outage_type": ["POWER", "HARDWARE", "BACKHAUL"],
                       "severity": ["LOW", "MEDIUM", "HIGH", "CRITICAL"],
                       "incident_status": ["OPEN", "INVESTIGATING", "RESOLVED"]},
        "time_order": [("outage_end_ts", "outage_start_ts", "END_BEFORE_START")],
    },
    "complaints": {
        "bronze": "bronze_customer_complaints", "silver": "silver_customer_complaints", "key": "complaint_id",
        "mandatory": ["complaint_id", "tower_id", "complaint_ts"],
        "refs": [("tower_id", "tower", "tower_id", "INVALID_TOWER_REFERENCE"),
                 ("region_id", "region", "region_id", "INVALID_REGION_REFERENCE")],
        "casts": {"complaint_ts": "TIMESTAMP", "resolved_ts": "TIMESTAMP"},
        "categories": {"complaint_type": ["NO_SIGNAL", "SLOW_DATA", "DROPPED_CALL"],
                       "severity": ["LOW", "MEDIUM", "HIGH", "CRITICAL"],
                       "ticket_status": ["OPEN", "IN_PROGRESS", "CLOSED"],
                       "channel": ["ECOM", "POS", "ATM"]},
        "time_order": [("resolved_ts", "complaint_ts", "RESOLVED_BEFORE_COMPLAINT")],
    },
}

OUTLIER_IQR_K = 1.5   # Tukey's standard fence: below Q1 - 1.5*IQR or above Q3 + 1.5*IQR
issue_frames = []     # filled by run_dataset; written to the rejected / warning tables below

In [0]:
# 3b. DQ rule catalogue (generated from DATASETS, so it always matches the checks the engine runs)
# rule_id = <DATASET>-<CATEGORY>-<NNN>; validation_query counts violations directly in Bronze (independent of the engine).
catalogue_table = f"{catalog}.{silver_schema}.silver_dq_rule_catalogue"

DATASET_CODES = {"region": "REG", "tower": "TWR", "topology": "TOP", "kpi": "KPI", "outage": "OUT", "complaints": "CMP"}
CATEGORY_CODES = {"NULL_CHECK": "NUL", "DUPLICATE": "DUP", "DATA_TYPE": "TYP", "NUMERIC_RANGE": "RNG",
                  "CATEGORICAL": "CAT", "TIMESTAMP": "TSP", "REFERENTIAL": "REF", "OUTLIER": "OTL"}
# Severity: CRITICAL = record identity broken, HIGH = record unusable, MEDIUM = usable but suspicious, LOW = informational
SEVERITY = {"MISSING_BUSINESS_KEY": "CRITICAL", "EXACT_DUPLICATE_ROW": "CRITICAL", "DUPLICATE_BUSINESS_KEY": "CRITICAL",
            "MISSING_OPTIONAL_FIELD": "LOW"}
ACTIONS = {REJECT: "Exclude from Silver; write to silver_rejected_records",
           WARN: "Keep in Silver; flag in dq_warning_checks and silver_warning_records"}

def _s(c, alias=""):
    """Trimmed text value of a Bronze column, blank -> NULL (same standardisation as the engine)."""
    prefix = f"{alias}." if alias else ""
    return f"NULLIF(TRIM(CAST({prefix}`{c}` AS STRING)), '')"

def build_rules(name, cfg):
    T = f"{catalog}.{bronze_schema}.{cfg['bronze']}"
    S = f"{catalog}.{silver_schema}.{cfg['silver']}"
    src = [c for c in spark.table(T).columns if c not in AUDIT_COLS]
    key, casts = cfg["key"], cfg.get("casts", {})
    count = f"SELECT COUNT(*) AS violations FROM {T}"
    row_hash = "sha2(concat_ws('||', " + ", ".join(f"COALESCE(CAST(`{c}` AS STRING), '<NULL>')" for c in src) + "), 256)"
    rules = []

    def add(category, check, column, level, description, query):
        rules.append({"dataset": name, "source_table": cfg["bronze"], "target_table": cfg["silver"],
                      "rule_category": category, "check_name": check, "column_name": column,
                      "rule_description": description, "issue_level": level,
                      "severity": SEVERITY.get(check, "HIGH" if level == REJECT else "MEDIUM"),
                      "handling_action": ACTIONS[level], "validation_query": " ".join(query.split())})

    # 1. Null checks
    for c in src:
        if c == key:
            add("NULL_CHECK", "MISSING_BUSINESS_KEY", c, REJECT, f"Business key {c} is missing or blank", f"{count} WHERE {_s(c)} IS NULL")
        elif c in cfg.get("mandatory", []):
            add("NULL_CHECK", "MISSING_MANDATORY_FIELD", c, REJECT, f"Mandatory field {c} is missing or blank", f"{count} WHERE {_s(c)} IS NULL")
        else:
            add("NULL_CHECK", "MISSING_OPTIONAL_FIELD", c, WARN, f"Optional field {c} is missing or blank", f"{count} WHERE {_s(c)} IS NULL")
    # 2. Duplicates
    add("DUPLICATE", "EXACT_DUPLICATE_ROW", key, REJECT,
        "Exact copy of an earlier delivered row (all source columns identical); first delivery kept",
        f"SELECT COUNT(*) - COUNT(DISTINCT {row_hash}) AS violations FROM {T}")
    add("DUPLICATE", "DUPLICATE_BUSINESS_KEY", key, REJECT,
        f"Older version of the same {key} with different content; latest ingested version kept",
        f"""SELECT COALESCE(SUM(n - 1), 0) AS violations FROM (
              SELECT {_s(key)} AS k, COUNT(DISTINCT {row_hash}) AS n FROM {T}
              WHERE {_s(key)} IS NOT NULL GROUP BY {_s(key)})""")
    # 3. Data types (DATE / TIMESTAMP conversions are timestamp rules)
    for c, t in casts.items():
        is_time = t.upper() in ("DATE", "TIMESTAMP")
        add("TIMESTAMP" if is_time else "DATA_TYPE", "INVALID_TIMESTAMP" if is_time else f"INVALID_{t.upper()}", c, REJECT,
            f"{c} cannot be converted to {t}",
            f"{count} WHERE {_s(c)} IS NOT NULL AND try_cast({_s(c)} AS {t}) IS NULL")
    # 4. Numeric ranges
    for c in cfg.get("non_negative", []):
        add("NUMERIC_RANGE", "NEGATIVE_VALUE", c, REJECT, f"{c} is negative",
            f"{count} WHERE try_cast({_s(c)} AS {casts[c]}) < 0")
    for c in cfg.get("percent", []):
        add("NUMERIC_RANGE", "PERCENT_OUT_OF_RANGE", c, REJECT, f"{c} is outside 0-100",
            f"{count} WHERE try_cast({_s(c)} AS {casts[c]}) NOT BETWEEN 0 AND 100")
    # 5. Categorical domains
    for c, allowed in cfg.get("categories", {}).items():
        values = ", ".join(f"'{v}'" for v in allowed)
        add("CATEGORICAL", "UNEXPECTED_CATEGORY", c, WARN, f"{c} not in observed domain ({', '.join(allowed)})",
            f"{count} WHERE UPPER({_s(c)}) NOT IN ({values})")
    # 6. Timestamp order
    for later, earlier, check in cfg.get("time_order", []):
        add("TIMESTAMP", check, later, REJECT, f"{later} is earlier than {earlier}",
            f"{count} WHERE try_cast({_s(later)} AS TIMESTAMP) < try_cast({_s(earlier)} AS TIMESTAMP)")
    # 7. Referential integrity against the clean Silver masters
    for c, ref_ds, ref_key, check in cfg.get("refs", []):
        M = f"{catalog}.{silver_schema}.{DATASETS[ref_ds]['silver']}"
        add("REFERENTIAL", check, c, REJECT, f"{c} not found in {DATASETS[ref_ds]['silver']}",
            f"SELECT COUNT(*) AS violations FROM {T} b LEFT ANTI JOIN {M} m ON {_s(c, 'b')} = m.`{ref_key}` WHERE {_s(c, 'b')} IS NOT NULL")
    # KPI-specific checks (implemented in kpi_extra_checks)
    if name == "kpi":
        file_stats = (f"(SELECT _source_file AS src, MIN(_ingested_at) AS ing, "
                      f"MAX(try_cast({_s('measurement_ts')} AS TIMESTAMP)) AS mx FROM {T} GROUP BY _source_file)")
        add("TIMESTAMP", "LATE_OUT_OF_ORDER", "measurement_ts", WARN,
            "measurement_ts is earlier than the latest measurement already delivered in previous batch files",
            f"""SELECT COUNT(*) AS violations FROM {T} b JOIN (
                  SELECT f.src, MAX(e.mx) AS prev_max FROM {file_stats} f JOIN {file_stats} e
                    ON e.ing < f.ing OR (e.ing = f.ing AND e.src < f.src)
                  GROUP BY f.src) p ON b._source_file = p.src
                WHERE try_cast({_s('measurement_ts', 'b')} AS TIMESTAMP) < p.prev_max""")
        TOP = f"{catalog}.{silver_schema}.{DATASETS['topology']['silver']}"
        add("REFERENTIAL", "CELL_TOWER_MISMATCH", "cell_id", WARN,
            "cell_id belongs to a different tower in silver_network_topology_master",
            f"SELECT COUNT(*) AS violations FROM {T} b JOIN {TOP} t ON {_s('cell_id', 'b')} = t.cell_id "
            f"WHERE {_s('tower_id', 'b')} <> t.tower_id")
    # 8. Outliers (Tukey fences on records that passed every REJECT rule = the Silver table)
    for m in cfg.get("outlier_metrics", []):
        k, v = OUTLIER_IQR_K, f"try_cast({_s(m, 'b')} AS {casts[m]})"
        add("OUTLIER", "IQR_OUTLIER", m, WARN, f"{m} outside Q1 - {k}*IQR / Q3 + {k}*IQR (fences from valid records)",
            f"""SELECT COUNT(*) AS violations FROM {T} b CROSS JOIN (
                  SELECT percentile_approx(`{m}`, array(0.25, 0.75), 10000) AS q FROM {S}) p
                WHERE {v} < p.q[0] - {k} * (p.q[1] - p.q[0]) OR {v} > p.q[1] + {k} * (p.q[1] - p.q[0])""")

    # Number the rules within dataset + category in a fixed order
    seq = {}
    for r in rules:
        grp = (DATASET_CODES[name], CATEGORY_CODES[r["rule_category"]])
        seq[grp] = seq.get(grp, 0) + 1
        r["rule_id"] = f"{grp[0]}-{grp[1]}-{seq[grp]:03d}"
    return rules

CATALOGUE_COLS = ["rule_id", "dataset", "source_table", "target_table", "rule_category", "check_name", "column_name",
                  "rule_description", "severity", "issue_level", "handling_action", "validation_query"]
catalogue_rows = [r for name, cfg in DATASETS.items() for r in build_rules(name, cfg)]
catalogue_df = spark.createDataFrame([tuple(r[c] for c in CATALOGUE_COLS) for r in catalogue_rows],
                                     ", ".join(f"{c} string" for c in CATALOGUE_COLS))

# Each rule must be unique, so every detected issue maps to exactly one rule_id
rule_ids = [r["rule_id"] for r in catalogue_rows]
rule_keys_list = [(r["source_table"], r["check_name"], r["column_name"]) for r in catalogue_rows]
assert len(set(rule_ids)) == len(rule_ids), "duplicate rule_id in catalogue"
assert len(set(rule_keys_list)) == len(rule_keys_list), "duplicate (table, check, column) in catalogue"

(catalogue_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(catalogue_table))
print(f"Saved {catalogue_table}: {len(catalogue_rows)} rules")
display(catalogue_df.drop("validation_query").orderBy("rule_id"))

In [0]:
# 4. Master and reference data (processed first, so facts are checked against the clean masters)
# Reference checks are defined in DATASETS[...]["refs"]
run_dataset("region")
run_dataset("tower")
run_dataset("topology")

In [0]:
# 5. Network KPI measurements
def kpi_extra_checks(df):
    # Late / out-of-order: measurement earlier than the latest measurement already delivered in previous files
    # One row per file (delivery order = first ingestion time, then file name). A small self-join replaces
    # an unpartitioned window, so all KPI rows are not moved to a single partition.
    files = df.groupBy("_lineage").agg(F.min("_ingested_at").alias("_f_ing"), F.max("measurement_ts").alias("_f_max"))
    earlier = files.select(F.col("_lineage").alias("_e_lin"), F.col("_f_ing").alias("_e_ing"), F.col("_f_max").alias("_e_max"))
    delivered_before = (F.col("_e_ing") < F.col("_f_ing")) | ((F.col("_e_ing") == F.col("_f_ing")) & (F.col("_e_lin") < F.col("_lineage")))
    files = (files.join(F.broadcast(earlier), delivered_before, "left")
                  .groupBy("_lineage").agg(F.max("_e_max").alias("_prev_files_max_ts")))
    df = df.join(F.broadcast(files), "_lineage", "left")

    # Cell must belong to the tower reported on the measurement
    cells = silver_table("topology").select(F.col("cell_id").alias("_c_cell"), F.col("tower_id").alias("_c_tower"))
    df = df.join(F.broadcast(cells), F.col("cell_id") == F.col("_c_cell"), "left")

    return df, [
        issue(WARN, "TIMESTAMP", "LATE_OUT_OF_ORDER", "measurement_ts",
              F.col("measurement_ts") < F.col("_prev_files_max_ts")),
        issue(WARN, "REFERENTIAL", "CELL_TOWER_MISMATCH", "cell_id",
              F.col("_c_cell").isNotNull() & (F.col("tower_id") != F.col("_c_tower"))),
    ]

run_dataset("kpi", extra=kpi_extra_checks)

In [0]:
# 6. Network outage events and customer complaints
# Reference checks are defined in DATASETS[...]["refs"]
run_dataset("outage")
run_dataset("complaints")

In [0]:
# 7. Quarantine: rejected records -> silver_rejected_records, warnings -> silver_warning_records
# Both tables are overwritten, so they always match the current Silver tables (no run history).
issues_all = reduce(lambda a, b: a.unionByName(b), issue_frames)

# Attach the catalogue rule_id / severity to every detected issue; every issue must map to exactly one rule
rule_keys = catalogue_df.select("source_table", "check_name", F.col("column_name").alias("issue_column"),
                                "rule_id", "severity")
issues_all = issues_all.join(F.broadcast(rule_keys), ["source_table", "check_name", "issue_column"], "left")
unmapped = issues_all.filter(F.col("rule_id").isNull()).count()
assert unmapped == 0, f"{unmapped} detected issues have no rule in the DQ rule catalogue"

rejected_out = (issues_all.filter((F.col("record_status") == "REJECTED") & (F.col("issue_level") == REJECT))
    .select("rule_id", "severity", "source_table", "target_table", "business_key_column", "business_key_value",
            F.col("check_name").alias("reject_reason"),
            F.col("issue_column").alias("rejected_column"),
            F.col("issue_value").alias("rejected_value"),
            F.lit("EXCLUDED_FROM_SILVER").alias("silver_action"),
            "source_ingested_at", "source_record_json",
            F.col("checked_at").alias("quarantined_at"),
            "issue_category", "issue_level"))

warnings_out = (issues_all.filter((F.col("record_status") == "VALID") & (F.col("issue_level") == WARN))
    .select("rule_id", "severity", "source_table", "target_table", "business_key_column", "business_key_value",
            F.col("check_name").alias("warning_reason"),
            F.col("issue_column").alias("warning_column"),
            F.col("issue_value").alias("warning_value"),
            F.lit("KEPT_IN_SILVER").alias("silver_action"),
            "source_ingested_at", "source_record_json",
            F.col("checked_at").alias("flagged_at"),
            "issue_category", "issue_level"))

for out_df, target in [(rejected_out, rejected_table), (warnings_out, warning_table)]:
    (out_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(target))
    print(f"Saved {target}")

# Per-rule detection counts of this run -> silver_dq_rule_results (overwritten each run).
# Used by tests/06_Pipeline_Validation to reconcile each rule: Bronze validation query = detected = stored.
detected_by_rule = issues_all.groupBy("rule_id").agg(
    F.count("*").alias("detected_issues"),
    F.sum(F.when(F.col("record_status") == "REJECTED", 1).otherwise(0)).alias("detected_on_rejected_records"),
    F.sum(F.when(F.col("record_status") == "VALID", 1).otherwise(0)).alias("detected_on_valid_records"))
rule_results_df = (catalogue_df.select("rule_id", "dataset", "source_table", "rule_category", "check_name",
                                       "column_name", "severity", "issue_level")
    .join(detected_by_rule, "rule_id", "left")
    .fillna(0, ["detected_issues", "detected_on_rejected_records", "detected_on_valid_records"])
    .withColumn("checked_at", F.current_timestamp()))
save_silver(rule_results_df, "silver_dq_rule_results")